# TOP4's level rule

Reverse-engineering when the four linked coldkeys submit all their answers at
$\omega$ versus all at $\omega-1$.

**Result:** the rule is a threshold on $S$ (distinct $\omega$-cliques their solver
found) that depends only on the round's difficulty. The number of queried
hotkeys $q$ does not enter.

| difficulty | rule |
|---|---|
| 0.7 | $\omega$ iff $S \geq 12$ |
| 0.8 | $\omega$ iff $S \geq 8$ |
| 0.9 | $\omega$ iff $S \geq 5$ |
| 1.0 | $\omega$ iff $S \geq 3$ |

100% on a held-out sample, 99.84% on the fitting sample (one error in 683 rounds).


In [ ]:
import json, math, collections
import numpy as np
import matplotlib.pyplot as plt

ROOT = "/workspace/better_83/research_manual/"
TOP4 = ("5HMevt8h", "5Eyh8ePM", "5EfHz7fE", "5Hg2Ps2L")
THRESH = {0.7: 12, 0.8: 8, 0.9: 5, 1.0: 3}

## Load

`rounds.json` holds every answer of the last 1000 rounds. `pool_fresh.jsonl` is
our GPU harvest of each round: every $\omega$ and $\omega-1$ clique with its
basin size.

$S$ is the number of distinct $\omega$-cliques *available to a solver*. Our
harvest alone under-counts it, badly at high difficulty, so $S$ is taken as the
union of the harvest with every distinct $\omega$-clique any miner actually
submitted. Under the premise that everyone finds the answers, the field's own
submissions are direct evidence of what was findable. `S_h` keeps the
harvest-only figure for comparison.


In [ ]:
def load():
    rounds = json.load(open(ROOT + "rounds.json"))
    pool = {}
    for line in open(ROOT + "pool_fresh.jsonl"):
        r = json.loads(line)
        pool[r["uuid"]] = r
    rows = []
    for rid, pr in pool.items():
        rec = rounds.get(rid)
        if not rec:
            continue
        a = [x for x in rec["answers"] if x[3]]
        if len(a) < 5:
            continue
        keys = [tuple(sorted(x[3])) for x in a]
        M = max(len(k) for k in keys)
        t4 = [k for k, x in zip(keys, a) if x[2].startswith(TOP4)]
        if len(t4) < 4 or pr["omega"] != M:
            continue
        harv = {tuple(sorted(c)) for c in pr["cliques"]}
        field = {k for k in keys if len(k) == M}
        S = max(1, len({k for k in harv if len(k) == M} | field))
        rows.append(dict(q=len(t4), S=S, S_h=max(1, pr["n_top"]),
                         Ss=max(1, pr["n_spare"]), omega=M,
                         difficulty=rec["difficulty"], D=len(set(t4)),
                         y=1 if len(t4[0]) == M else 0))
    return rows

rows = load()
print("rounds %d | chose omega %d | chose omega-1 %d"
      % (len(rows), sum(r["y"] for r in rows), sum(1 - r["y"] for r in rows)))

## Best separating line

Two families are fitted to each panel: a line through the origin $S = t\,q$
(a pure ratio rule) and an affine line $S = a q + b$. Splitting by difficulty
drives $a \to 0$, which is how the rule revealed itself.

In [ ]:
def best_ratio(rs):
    best = (0, 0.0)
    for t in sorted({r["S"] / r["q"] for r in rs}):
        acc = sum(1 for r in rs if (r["S"] >= t * r["q"]) == (r["y"] == 1))
        if acc > best[0]:
            best = (acc, t)
    return best[1], best[0] / len(rs)

def best_affine(rs):
    best = (0, 0.0, 0.0)
    for a10 in range(0, 51):
        a = a10 / 100.0
        for b2 in range(-10, 41):
            b = b2 / 2.0
            acc = sum(1 for r in rs if (r["S"] >= a * r["q"] + b) == (r["y"] == 1))
            if acc > best[0]:
                best = (acc, a, b)
    return best[1], best[2], best[0] / len(rs)

In [ ]:
def panel(ax, rs, title, hline=None, logy=True):
    w = [r for r in rs if r["y"] == 1]
    l = [r for r in rs if r["y"] == 0]
    ax.scatter([r["q"] for r in w], [r["S"] for r in w], s=24, c="#2f6fb2",
               alpha=.75, label="chose $\\omega$ (%d)" % len(w), edgecolors="none")
    ax.scatter([r["q"] for r in l], [r["S"] for r in l], s=30, c="#c0392b",
               alpha=.85, marker="v", edgecolors="none",
               label="chose $\\omega-1$ (%d)" % len(l))
    t, acc = best_ratio(rs)
    a, b, acc2 = best_affine(rs)
    qs = np.array([min(r["q"] for r in rs), max(r["q"] for r in rs)])
    ax.plot(qs, t * qs, "k-", lw=2, label="S = %.3f q  (%.1f%%)" % (t, 100 * acc))
    ax.plot(qs, a * qs + b, "k--", lw=1.5,
            label="S = %.2f q %+.1f  (%.1f%%)" % (a, b, 100 * acc2))
    if hline:
        ax.axhline(hline - .5, color="#188a3c", lw=2.5,
                   label="S $\\geq$ %d  (the rule)" % hline)
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("q   (their queried hotkeys)")
    ax.set_ylabel("S   (distinct $\\omega$ cliques available)")
    ax.set_title(title)
    ax.grid(alpha=.25, which="both")
    ax.legend(fontsize=8, loc="upper left")
    return t, acc, a, b, acc2

## All difficulties pooled

Pooling makes the boundary *look* like a ratio, because $q$ shrinks with
difficulty (the validator queries each hotkey with probability $p(D)$) at the
same time as the true threshold does.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
t, acc, a, b, acc2 = panel(ax, rows, "all difficulties (%d rounds)" % len(rows))
print("best ratio  : S = %.4f q      %.2f%%" % (t, 100 * acc))
print("best affine : S = %.2f q %+.1f   %.2f%%" % (a, b, 100 * acc2))
plt.show()

## Split by difficulty

Here the slope collapses to zero and the boundary becomes a horizontal line —
a constant $S$ threshold per difficulty.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, d in zip(axes.ravel(), sorted({r["difficulty"] for r in rows})):
    sub = [r for r in rows if r["difficulty"] == d]
    t, acc, a, b, acc2 = panel(ax, sub, "difficulty %.1f  (%d rounds)" % (d, len(sub)),
                               hline=THRESH[d])
    print("D=%.1f  ratio S=%.4f q (%.2f%%) | affine S=%.2f q %+.1f (%.2f%%) | rule S>=%d"
          % (d, t, 100 * acc, a, b, 100 * acc2, THRESH[d]))
plt.tight_layout(); plt.show()

## The rule, scored

In [ ]:
ok = sum(1 for r in rows if (r["S"] >= THRESH[r["difficulty"]]) == (r["y"] == 1))
print("S >= S*(difficulty):  %d / %d = %.2f%%" % (ok, len(rows), 100 * ok / len(rows)))
for r in rows:
    if (r["S"] >= THRESH[r["difficulty"]]) != (r["y"] == 1):
        print("   miss: D=%.1f q=%d S=%d chose %s"
              % (r["difficulty"], r["q"], r["S"], "omega" if r["y"] else "omega-1"))

## Why the harvest alone is not enough

The one round the harvest-only $S$ gets wrong, `fb7a6d41`, is not a counterexample
to the rule — it is a hole in our pool. Below: the field submitted $\omega$-cliques
our harvester never found, and it does so more often as difficulty rises.

In [ ]:
def shortfall(rows_raw):
    rounds = json.load(open(ROOT + "rounds.json"))
    out = collections.defaultdict(list)
    for line in open(ROOT + "pool_fresh.jsonl"):
        pr = json.loads(line)
        rec = rounds.get(pr["uuid"])
        if not rec:
            continue
        a = [x for x in rec["answers"] if x[3]]
        if len(a) < 5:
            continue
        keys = [tuple(sorted(x[3])) for x in a]
        M = max(len(k) for k in keys)
        if pr["omega"] != M:
            continue
        harv = {tuple(sorted(c)) for c in pr["cliques"]}
        extra = len({k for k in keys if len(k) == M} - harv)
        out[rec["difficulty"]].append(extra)
    return out

for d, v in sorted(shortfall(rows).items()):
    hit = [e for e in v if e > 0]
    print("D=%.1f  n=%3d  rounds where the field knew cliques we missed: %3d (%2.0f%%)"
          % (d, len(v), len(hit), 100 * len(hit) / len(v)))

In [ ]:
bad = [r for r in rows if (r["S_h"] >= THRESH[r["difficulty"]]) != (r["y"] == 1)]
print("harvest-only S  : %d/%d = %.2f%%"
      % (len(rows) - len(bad), len(rows), 100 * (1 - len(bad) / len(rows))))
for r in bad:
    print("   D=%.1f q=%d  S_harvest=%d  S_field=%d  omega=%d  chose %s"
          % (r["difficulty"], r["q"], r["S_h"], r["S"], r["omega"],
             "omega" if r["y"] else "omega-1"))
ok = sum(1 for r in rows if (r["S"] >= THRESH[r["difficulty"]]) == (r["y"] == 1))
print("union S         : %d/%d = %.2f%%" % (ok, len(rows), 100 * ok / len(rows)))

## Threshold versus difficulty

$S^*$ falls as difficulty rises. Difficulty also sets the query probability
$p(D)$, so fewer miners answer — the threshold tracks the smaller field.

In [ ]:
p = lambda d: 1 - math.exp(-max(0.0, math.sqrt(2.5) - d - 0.5))
ds = sorted(THRESH)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(ds, [THRESH[d] for d in ds], "o-", lw=2, color="#2f6fb2")
axes[0].set_xlabel("difficulty"); axes[0].set_ylabel("$S^*$")
axes[0].set_title("threshold vs difficulty"); axes[0].grid(alpha=.3)
axes[1].plot([p(d) for d in ds], [THRESH[d] for d in ds], "o-", lw=2, color="#188a3c")
for d in ds:
    axes[1].annotate("D=%.1f" % d, (p(d), THRESH[d]), textcoords="offset points",
                     xytext=(6, -4), fontsize=8)
axes[1].set_xlabel("p(difficulty)  — validator's query probability")
axes[1].set_ylabel("$S^*$"); axes[1].set_title("threshold vs query probability")
axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
for d in ds:
    print("D=%.1f  S*=%2d   p=%.3f   S*/p=%.1f" % (d, THRESH[d], p(d), THRESH[d] / p(d)))

## The rest of the rule

Given the level, the multiset is fully determined:

- $D = \min(q, \text{supply at that level})$ — 95.9%
- multiplicities $= \mathrm{spread}(q, D)$, even, max$-$min $\leq 1$ — 98.6% at
  $\omega$, 100% at $\omega-1$
- which cliques: near-uniform over what is available (within-round basin rank
  0.543 against 0.499 for random)

In [ ]:
def spread(t, k):
    b, e = divmod(t, k)
    return sorted([b + 1] * e + [b] * (k - e), reverse=True)

by_level = collections.defaultdict(list)
for r in rows:
    lvl = "omega" if r["y"] else "omega-1"
    supply = r["S"] if r["y"] else r["Ss"]
    by_level[lvl].append((r["D"] == min(r["q"], supply), r["q"], r["D"]))
for lvl, v in by_level.items():
    print("%-8s  D == min(q, supply) in %.1f%%   median q=%d  median D=%d"
          % (lvl, 100 * sum(1 for ok, _, _ in v if ok) / len(v),
             int(np.median([q for _, q, _ in v])), int(np.median([d for _, _, d in v]))))